# FlightAI — adding a `set_ticket_price` tool

Week 2, Day 4 exercise: extend the airline assistant with a second tool so the user can *set* a ticket price, not just ask for one. Prices live in SQLite (`prices.db`, created by this notebook), so a price set in one message is still there in the next.

Try in the chat at the end:
- "What's the price of a ticket to Tokyo?"
- "Set the price of a ticket to Tokyo to $1200"
- "How much is a ticket to Tokyo now?" 

In [ ]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

MODEL = "gpt-4.1-mini"
openai = OpenAI()

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
You can check ticket prices with get_ticket_price, and you can change a ticket price with set_ticket_price when the customer asks you to set, update, or change a price. After setting a price, confirm the new price to the customer.
"""

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} to {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"The price of a ticket to {city} has been set to ${price:g}"

In [ ]:
for city, price in {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}.items():
    set_ticket_price(city, price)

get_ticket_price("Tokyo")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city. Use this when the customer asks to change, update, or set a ticket price.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price, in dollars, e.g. 750 or 899.99",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        if tool_call.function.name == "get_ticket_price":
            city = arguments.get('destination_city')
            result = get_ticket_price(city)
        elif tool_call.function.name == "set_ticket_price":
            city = arguments.get('destination_city')
            price = arguments.get('price')
            result = set_ticket_price(city, price)
        else:
            result = f"Unknown tool: {tool_call.function.name}"
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
# Quick check without the LLM: set a new city's price, then read it back
print(set_ticket_price("Berlin", 499))
print(get_ticket_price("Berlin"))

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()